# 04 – SQL analysis
Runs the SQL files in `sql/` to build change-over-time, ranking, and Mobility
Barriers Index views, explores key findings, and exports tables for Tableau.

In [1]:
import duckdb

con = duckdb.connect("../data/processed/mobility.duckdb")

def run_sql_file(filename):
    with open(f"../sql/{filename}") as f:
        con.execute(f.read())
    print(f"Ran {filename}")

In [2]:
run_sql_file("01_change_over_time.sql")
run_sql_file("02_rankings.sql")
run_sql_file("03_mobility_index.sql")

Ran 01_change_over_time.sql
Ran 02_rankings.sql
Ran 03_mobility_index.sql


In [3]:
con.sql("""
    SELECT year,
           COUNT(*) AS counties,
           ROUND(AVG(barrier_index), 2) AS avg_index
    FROM mobility_index
    GROUP BY year
""").show()

┌───────┬──────────┬───────────┐
│ year  │ counties │ avg_index │
│ int64 │  int64   │  double   │
├───────┼──────────┼───────────┤
│  2019 │       24 │      -0.0 │
│  2024 │       24 │      -0.0 │
└───────┴──────────┴───────────┘



In [4]:
con.sql("""
    SELECT county_name, rent_burden_rate, median_income_2024usd
    FROM county_indicators
    WHERE year = 2024
    ORDER BY rent_burden_rate DESC
    LIMIT 5
""").show()

┌─────────────────────┬──────────────────┬───────────────────────┐
│     county_name     │ rent_burden_rate │ median_income_2024usd │
│       varchar       │      double      │        double         │
├─────────────────────┼──────────────────┼───────────────────────┤
│ Kent County         │            59.63 │               80147.0 │
│ Dorchester County   │             56.0 │               61839.0 │
│ Caroline County     │            55.78 │               68457.0 │
│ Worcester County    │            55.32 │               81745.0 │
│ Queen Anne's County │            54.38 │              112826.0 │
└─────────────────────┴──────────────────┴───────────────────────┘



In [5]:
con.sql("""
    SELECT county_name,
           income_change_pct,
           rent_burden_change_pts,
           CASE
               WHEN income_change_pct < 0 AND rent_burden_change_pts > 0
                   THEN 'Squeezed: income down, rent burden up'
               WHEN rent_burden_change_pts > 0
                   THEN 'Rent burden up'
               ELSE 'Rent burden flat or down'
           END AS housing_pressure
    FROM county_change
    ORDER BY rent_burden_change_pts DESC
""").show()

┌────────────────────────┬───────────────────┬────────────────────────┬───────────────────────────────────────┐
│      county_name       │ income_change_pct │ rent_burden_change_pts │           housing_pressure            │
│        varchar         │      double       │         double         │                varchar                │
├────────────────────────┼───────────────────┼────────────────────────┼───────────────────────────────────────┤
│ Queen Anne's County    │              -5.2 │                    6.2 │ Squeezed: income down, rent burden up │
│ Worcester County       │               4.9 │                    5.2 │ Rent burden up                        │
│ Baltimore County       │              -2.7 │                    4.7 │ Squeezed: income down, rent burden up │
│ Caroline County        │              -4.9 │                    4.7 │ Squeezed: income down, rent burden up │
│ Howard County          │               0.7 │                    4.3 │ Rent burden up                  

In [6]:
con.sql("""
    SELECT county_name, barrier_index, barrier_rank, barrier_tier
    FROM mobility_index
    WHERE year = 2024
    ORDER BY barrier_rank
""").show()

┌────────────────────────┬───────────────┬──────────────┬─────────────────┐
│      county_name       │ barrier_index │ barrier_rank │  barrier_tier   │
│        varchar         │    double     │    int64     │     varchar     │
├────────────────────────┼───────────────┼──────────────┼─────────────────┤
│ Baltimore city         │          1.06 │            1 │ High barriers   │
│ Dorchester County      │          1.05 │            2 │ High barriers   │
│ Somerset County        │          0.85 │            3 │ High barriers   │
│ Allegany County        │          0.82 │            4 │ High barriers   │
│ Caroline County        │          0.82 │            5 │ High barriers   │
│ Kent County            │          0.62 │            6 │ High barriers   │
│ Wicomico County        │          0.51 │            7 │ High barriers   │
│ Prince George's County │          0.44 │            8 │ High barriers   │
│ Worcester County       │          0.43 │            9 │ Medium barriers │
│ Washington

In [7]:
con.sql("""
    SELECT
        ROUND(CORR(broadband_rate, poverty_rate), 2)      AS broadband_vs_poverty,
        ROUND(CORR(bachelors_plus_rate, poverty_rate), 2) AS education_vs_poverty
    FROM county_indicators
    WHERE year = 2024
""").show()

┌──────────────────────┬──────────────────────┐
│ broadband_vs_poverty │ education_vs_poverty │
│        double        │        double        │
├──────────────────────┼──────────────────────┤
│                -0.77 │                -0.65 │
└──────────────────────┴──────────────────────┘



## Findings so far
- [County] has the highest rent burden: X% of renters pay 30%+ of income on rent.
- In X counties, real income fell while rent burden rose.
- Highest-barrier counties: ...
- Broadband access and poverty are [strongly/moderately] correlated (r = X).

In [8]:
con.sql("""
    COPY (
        SELECT i.*, m.barrier_index, m.barrier_rank, m.barrier_tier
        FROM county_indicators AS i
        JOIN mobility_index AS m USING (fips, year)
        ORDER BY i.year, i.county_name
    ) TO '../data/processed/tableau_county_year.csv' (HEADER)
""")

con.sql("""
    COPY (SELECT * FROM county_change ORDER BY county_name)
    TO '../data/processed/tableau_county_change.csv' (HEADER)
""")

print("Exported Tableau files")

Exported Tableau files


In [9]:
con.close()